# TNG300-1: constraining descendant mass with high-redshift galaxy properties

This notebook starts from `tng300_z8_M11_descendants.csv`, produced by the preceding merger-tree analysis, and asks how much observable information at $z\simeq8$ narrows the simulation-derived conditional distribution of the snapshot-99 FoF host mass.

The features are deliberately limited to $M_{200c,z\simeq8}$, central stellar mass, central SFR, and three-dimensional galaxy richness. The phrase “posterior width” below is shorthand for the width of a **simulation-derived conditional descendant-mass distribution**, not a fully specified Bayesian posterior.

## Cell 1: imports

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial import cKDTree
from IPython.display import display, Markdown

import illustris_python as il

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

## Cell 2: paths and parameters

TNG masses use $10^{10}M_\odot/h$ and positions use comoving kpc/$h$. Snapshot 8 is the same $z\simeq8$ snapshot used by the preceding notebook. Header values verify the redshift, $h$, and periodic box size.

In [ ]:
base_candidates = [
    Path("/home/tnguser/sims.TNG/L205n2500TNG/output"),
    Path("../sims.TNG/TNG300-1/output").resolve(),
    Path("../sims.TNG/L205n2500TNG/output").resolve(),
]
basePath = next((p for p in base_candidates if p.is_dir()), None)
if basePath is None:
    raise FileNotFoundError(
        "TNG300-1 was not found. Set basePath to its output directory. Tried:\n"
        + "\n".join(str(p) for p in base_candidates)
    )

descendant_csv = Path.cwd() / "tng300_z8_M11_descendants.csv"
snap_z8 = 8
snap_z0 = 99
logM_min = 10.8
logM_max = 11.2

R_richness_cMpc = 5.0
logMstar_threshold = 8.0
exclude_central_from_richness = True
require_subhalo_flag = True
sfr_floor = 1.0e-3  # Msun/yr, used only for log10(SFR + floor)
random_state = 42
min_conditional_n = 30

header_z8 = il.groupcat.loadHeader(str(basePath), snap_z8)
z_z8 = float(header_z8["Redshift"])
h = float(header_z8["HubbleParam"])
box_size_ckpc_h = float(header_z8["BoxSize"])
box_size_cMpc = box_size_ckpc_h / (1000.0 * h)

assert np.isclose(h, 0.6774, atol=1e-4, rtol=0)
print(f"basePath              : {basePath}")
print(f"snapshot {snap_z8} redshift : {z_z8:.6f}")
print(f"snapshot z=0          : {snap_z0}")
print(f"h                     : {h:.4f}")
print(f"periodic box size     : {box_size_cMpc:.3f} cMpc")
print(f"richness definition   : R < {R_richness_cMpc:g} cMpc, log10(Mstar/Msun) > {logMstar_threshold:g}")

## Cell 3: load the existing descendant catalog

No merger trees are reloaded. The final mass remains the snapshot-99 FoF host `Group_M_Crit200` inherited from the preceding analysis, including objects that are satellites at $z=0$.

In [ ]:
required_columns = [
    "GroupID_z8", "SubhaloID_z8", "M200c_z8", "logM200c_z8",
    "SubhaloID_z0", "GroupID_z0", "M200c_z0", "logM200c_z0",
]
if not descendant_csv.is_file():
    raise FileNotFoundError(f"Run the preceding notebook first; missing {descendant_csv}")

raw_descendants = pd.read_csv(descendant_csv)
missing_columns = sorted(set(required_columns) - set(raw_descendants.columns))
if missing_columns:
    raise ValueError(f"Descendant catalog is missing columns: {missing_columns}")

finite_tracking = raw_descendants[required_columns].notna().all(axis=1)
mass_selection = (
    (raw_descendants["logM200c_z8"] > logM_min)
    & (raw_descendants["logM200c_z8"] < logM_max)
)
desc = raw_descendants.loc[finite_tracking & mass_selection, required_columns].copy()
for col in ["GroupID_z8", "SubhaloID_z8", "SubhaloID_z0", "GroupID_z0"]:
    desc[col] = desc[col].astype(np.int64)
desc.reset_index(drop=True, inplace=True)

print(f"Rows in descendant catalog : {len(raw_descendants):,}")
print(f"Successful finite rows     : {finite_tracking.sum():,}")
print(f"Final z≈8 mass-cut sample  : {len(desc):,}")
display(desc.head())

## Cell 4: load the snapshot-8 subhalo catalog once

The same full catalog supports central-galaxy properties and the later richness calculation. `SubhaloFlag == 1` identifies subhalos of cosmological origin and is used for the richness galaxy catalog when requested.

In [ ]:
subhalo_fields = [
    "SubhaloMassType", "SubhaloSFR", "SubhaloPos", "SubhaloGrNr", "SubhaloFlag"
]
subhalos_z8 = il.groupcat.loadSubhalos(str(basePath), snap_z8, fields=subhalo_fields)

subhalo_mass_type = np.asarray(subhalos_z8["SubhaloMassType"])
subhalo_sfr = np.asarray(subhalos_z8["SubhaloSFR"], dtype=float)
subhalo_pos_ckpc_h = np.asarray(subhalos_z8["SubhaloPos"], dtype=float)
subhalo_grnr = np.asarray(subhalos_z8["SubhaloGrNr"], dtype=np.int64)
subhalo_flag = np.asarray(subhalos_z8["SubhaloFlag"], dtype=bool)

target_sub_ids = desc["SubhaloID_z8"].to_numpy(dtype=np.int64)
if target_sub_ids.min() < 0 or target_sub_ids.max() >= len(subhalo_sfr):
    raise IndexError("A descendant-catalog SubhaloID_z8 is outside the snapshot-8 catalog.")

host_match = subhalo_grnr[target_sub_ids] == desc["GroupID_z8"].to_numpy()
if not np.all(host_match):
    raise ValueError(f"Central-to-host mismatch for {(~host_match).sum()} target objects")

print(f"All snapshot-8 subhalos loaded : {len(subhalo_sfr):,}")
print(f"Target central subhalos         : {len(target_sub_ids):,}")
print("All target central-to-FoF mappings verified.")

## Cell 5: central stellar mass

In the TNG six-component convention, stellar particles and wind-phase cells occupy PartType4, so the stellar component is column 4 of `SubhaloMassType`. The conversion is

$$M_\star[M_\odot] = M_\star[10^{10}M_\odot/h]\,10^{10}/h.$$

In [ ]:
stellar_parttype_index = 4
all_mstar_msun = subhalo_mass_type[:, stellar_parttype_index] * 1.0e10 / h
central_mstar = all_mstar_msun[target_sub_ids]

desc["Mstar_z8"] = central_mstar
desc["logMstar_z8"] = np.where(
    central_mstar > 0, np.log10(central_mstar), np.nan
)

print(f"Centrals with positive stellar mass: {desc['logMstar_z8'].notna().sum():,}/{len(desc):,}")
fig, ax = plt.subplots()
ax.hist(desc["logMstar_z8"].dropna(), bins=25, color="tab:blue", alpha=0.82)
ax.axvline(logMstar_threshold, color="k", ls="--", label="richness threshold")
ax.set(xlabel=r"$\log_{10}(M_{\star,\rm central}/M_\odot)$", ylabel="Number of target halos")
ax.legend()
plt.show()

## Cell 6: central SFR

`SubhaloSFR` is in $M_\odot\,\mathrm{yr}^{-1}$. Zero-SFR objects remain valid; `logSFR_z8` uses the explicitly stated numerical floor only for plotting and regression.

In [ ]:
desc["SFR_z8"] = subhalo_sfr[target_sub_ids]
desc["logSFR_z8"] = np.log10(desc["SFR_z8"] + sfr_floor)

print(f"Zero-SFR centrals: {(desc['SFR_z8'] == 0).sum():,}/{len(desc):,}")
fig, ax = plt.subplots()
ax.scatter(desc["logMstar_z8"], desc["logSFR_z8"], s=14, alpha=0.4, edgecolors="none")
ax.set(
    xlabel=r"$\log_{10}(M_{\star,\rm central}/M_\odot)$",
    ylabel=rf"$\log_{{10}}(\mathrm{{SFR}}+{sfr_floor:g})$",
)
plt.show()

## Cell 7: baseline $p(M_0\mid M_8)$

Because $M_8$ is already restricted to a narrow interval, the full successfully tracked sample is the baseline conditional distribution. $W_{68}=P_{84}-P_{16}$ is measured in dex.

In [ ]:
def summarize_distribution(values, label="sample"):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if values.size == 0:
        return pd.Series({
            "Label": label, "N": 0, "Median": np.nan, "P16": np.nan,
            "P84": np.nan, "W68": np.nan, "Std": np.nan, "IQR": np.nan,
        })
    p16, p25, p50, p75, p84 = np.percentile(values, [16, 25, 50, 75, 84])
    return pd.Series({
        "Label": label,
        "N": values.size,
        "Median": p50,
        "P16": p16,
        "P84": p84,
        "W68": p84 - p16,
        "Std": np.std(values, ddof=1) if values.size > 1 else np.nan,
        "IQR": p75 - p25,
    })


def grouped_distribution_stats(frame, group_col, value_col="logM200c_z0"):
    rows = []
    for name, group in frame.groupby(group_col, observed=True, sort=False):
        rows.append(summarize_distribution(group[value_col], str(name)))
    return pd.DataFrame(rows)


def rank_tertiles(series, labels=("low", "middle", "high")):
    """Balanced tertiles; ties at a boundary are split deterministically."""
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=3, labels=labels)


baseline_stats = summarize_distribution(desc["logM200c_z0"], "M8 only")
display(baseline_stats.to_frame().T.round(3))

fig, ax = plt.subplots()
ax.hist(desc["logM200c_z0"], bins=25, color="0.45", alpha=0.8)
ax.axvline(baseline_stats["Median"], color="k", lw=2, label=f"median={baseline_stats['Median']:.2f}")
ax.axvspan(baseline_stats["P16"], baseline_stats["P84"], color="tab:blue", alpha=0.15,
           label=f"W68={baseline_stats['W68']:.2f} dex")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Count")
ax.legend()
plt.show()

## Cell 8: condition on central stellar mass

The positive-stellar-mass sample is divided into rank-balanced tertiles. Rank balancing keeps the sample sizes comparable; if values tie exactly at a boundary, tied objects can fall on opposite sides of that boundary.

In [ ]:
analysis = desc.dropna(subset=["logMstar_z8", "logM200c_z0"]).copy()
analysis["Mstar_tertile"] = rank_tertiles(analysis["logMstar_z8"])
mstar_stats = grouped_distribution_stats(analysis, "Mstar_tertile")

fig, ax = plt.subplots()
common_bins = np.linspace(desc["logM200c_z0"].min(), desc["logM200c_z0"].max(), 27)
colors = {"low": "tab:blue", "middle": "tab:orange", "high": "tab:red"}
for label, group in analysis.groupby("Mstar_tertile", observed=True):
    ax.hist(group["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
            color=colors[str(label)], label=f"{label} Mstar (N={len(group)})")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
ax.legend()
plt.show()
display(mstar_stats.round(3))

## Cell 9: central stellar mass versus descendant mass

In [ ]:
pearson_mstar_r, pearson_mstar_p = stats.pearsonr(
    analysis["logMstar_z8"], analysis["logM200c_z0"]
)
spearman_mstar_r, spearman_mstar_p = stats.spearmanr(
    analysis["logMstar_z8"], analysis["logM200c_z0"]
)

analysis["Mstar_plot_bin"] = pd.qcut(
    analysis["logMstar_z8"].rank(method="first"), q=8, labels=False
)
binned_mstar = analysis.groupby("Mstar_plot_bin").agg(
    x=("logMstar_z8", "median"),
    y=("logM200c_z0", "median"),
    y16=("logM200c_z0", lambda x: np.percentile(x, 16)),
    y84=("logM200c_z0", lambda x: np.percentile(x, 84)),
    N=("logM200c_z0", "size"),
)

fig, ax = plt.subplots()
ax.scatter(analysis["logMstar_z8"], analysis["logM200c_z0"], s=12, alpha=0.28, edgecolors="none")
ax.errorbar(
    binned_mstar["x"], binned_mstar["y"],
    yerr=[binned_mstar["y"] - binned_mstar["y16"], binned_mstar["y84"] - binned_mstar["y"]],
    fmt="o-", color="k", capsize=3, label="bin median and 16–84%",
)
ax.set(
    xlabel=r"$\log_{10}(M_{\star,\rm central}/M_\odot)$",
    ylabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$",
)
ax.legend()
plt.show()
print(f"Pearson r  = {pearson_mstar_r:.3f} (p={pearson_mstar_p:.3e})")
print(f"Spearman ρ = {spearman_mstar_r:.3f} (p={spearman_mstar_p:.3e})")

## Cell 10: add central SFR

Two complementary summaries are used: global SFR tertiles for $p(M_0\mid M_8,\mathrm{SFR})$, and a low/high SFR split at the median **within each stellar-mass tertile** for $p(M_0\mid M_8,M_\star,\mathrm{SFR})$. This reduces confounding by the stellar-mass–SFR relation without invoking a multivariate model.

In [ ]:
analysis["SFR_tertile"] = rank_tertiles(analysis["logSFR_z8"])
sfr_stats = grouped_distribution_stats(analysis, "SFR_tertile")

analysis["SFR_split_within_Mstar"] = (
    analysis.groupby("Mstar_tertile", observed=True)["logSFR_z8"]
    .transform(lambda x: np.where(x <= x.median(), "low SFR", "high SFR"))
)
analysis["Mstar_SFR_cell"] = (
    analysis["Mstar_tertile"].astype(str) + " Mstar / " + analysis["SFR_split_within_Mstar"]
)
mstar_sfr_stats = grouped_distribution_stats(analysis, "Mstar_SFR_cell")

spearman_sfr_r, spearman_sfr_p = stats.spearmanr(
    analysis["logSFR_z8"], analysis["logM200c_z0"]
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].scatter(analysis["logSFR_z8"], analysis["logM200c_z0"], s=12, alpha=0.3, edgecolors="none")
axes[0].set(xlabel=rf"$\log_{{10}}(\mathrm{{SFR}}+{sfr_floor:g})$",
            ylabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$",
            title=rf"Spearman $\rho={spearman_sfr_r:.3f}$")

for split, color in [("low SFR", "tab:blue"), ("high SFR", "tab:red")]:
    subset = analysis[analysis["SFR_split_within_Mstar"] == split]
    axes[1].hist(subset["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
                 color=color, label=f"{split} (within Mstar tertiles; N={len(subset)})")
axes[1].set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

display(mstar_sfr_stats.round(3))
print(f"SFR Spearman ρ = {spearman_sfr_r:.3f} (p={spearman_sfr_p:.3e})")

## Cell 11: prepare the richness galaxy catalog

Snapshot positions are comoving kpc/$h$ and are converted to cMpc by dividing by $1000h$. Candidate galaxies have $M_\star>10^8M_\odot$ and, by default, `SubhaloFlag == 1`. Coordinates are wrapped into $[0,L)$ before building a periodic `cKDTree`.

In [ ]:
all_pos_cMpc = np.mod(subhalo_pos_ckpc_h / (1000.0 * h), box_size_cMpc)
galaxy_mask = np.isfinite(all_mstar_msun) & (all_mstar_msun > 10**logMstar_threshold)
if require_subhalo_flag:
    galaxy_mask &= subhalo_flag

richness_galaxy_ids = np.flatnonzero(galaxy_mask)
richness_galaxy_pos = all_pos_cMpc[richness_galaxy_ids]
target_pos_cMpc = all_pos_cMpc[target_sub_ids]

print(f"Richness candidate galaxies : {len(richness_galaxy_ids):,}")
print(f"Radius                       : {R_richness_cMpc:g} cMpc")
print(f"Stellar-mass threshold       : >10^{logMstar_threshold:g} Msun")
print(f"Require SubhaloFlag == 1     : {require_subhalo_flag}")
print(f"Periodic box size            : {box_size_cMpc:.3f} cMpc")

## Cell 12: calculate $N_{\rm gal}$

The periodic tree applies the minimum-image convention automatically. The target central is subtracted if it satisfies the galaxy selection, so `Ngal_R5_z8` counts neighboring galaxies and excludes the central itself.

In [ ]:
periodic_tree = cKDTree(richness_galaxy_pos, boxsize=box_size_cMpc)
try:
    neighbor_counts = periodic_tree.query_ball_point(
        target_pos_cMpc, r=R_richness_cMpc, return_length=True, workers=-1
    )
except TypeError:
    neighbor_lists = periodic_tree.query_ball_point(target_pos_cMpc, r=R_richness_cMpc)
    neighbor_counts = np.fromiter((len(x) for x in neighbor_lists), dtype=np.int64)

neighbor_counts = np.asarray(neighbor_counts, dtype=np.int64)
if exclude_central_from_richness:
    neighbor_counts -= galaxy_mask[target_sub_ids].astype(np.int64)
if np.any(neighbor_counts < 0):
    raise RuntimeError("Negative richness encountered after central subtraction")

desc["Ngal_R5_z8"] = neighbor_counts
analysis["Ngal_R5_z8"] = desc.loc[analysis.index, "Ngal_R5_z8"]

print(f"Central excluded: {exclude_central_from_richness}")
print(desc["Ngal_R5_z8"].describe(percentiles=[0.16, 0.5, 0.84]))
fig, ax = plt.subplots()
max_richness = int(desc["Ngal_R5_z8"].max())
richness_hist_bins = np.arange(max_richness + 2) - 0.5 if max_richness <= 100 else 40
ax.hist(desc["Ngal_R5_z8"], bins=richness_hist_bins,
        color="tab:green", alpha=0.82)
ax.set(xlabel=rf"$N_{{\rm gal}}(<{R_richness_cMpc:g}\,\rm cMpc)$ excluding central", ylabel="Target halos")
plt.show()

## Cell 13: richness versus descendant mass

Rank-balanced richness tertiles are used for comparable sample sizes. Since richness is discrete, tied richness values can be divided across a rank-tertile boundary; the actual value ranges and sample sizes are printed.

In [ ]:
analysis["Richness_tertile"] = rank_tertiles(analysis["Ngal_R5_z8"])
richness_stats = grouped_distribution_stats(analysis, "Richness_tertile")
richness_ranges = analysis.groupby("Richness_tertile", observed=True)["Ngal_R5_z8"].agg(["min", "max", "size"])
spearman_richness_r, spearman_richness_p = stats.spearmanr(
    analysis["Ngal_R5_z8"], analysis["logM200c_z0"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].scatter(analysis["Ngal_R5_z8"], analysis["logM200c_z0"], s=12, alpha=0.3, edgecolors="none")
axes[0].set(xlabel=r"$N_{\rm gal}$", ylabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$",
            title=rf"Spearman $\rho={spearman_richness_r:.3f}$")
for label, group in analysis.groupby("Richness_tertile", observed=True):
    axes[1].hist(group["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
                 label=f"{label} richness (N={len(group)})")
axes[1].set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
axes[1].legend()
plt.tight_layout()
plt.show()

display(richness_ranges)
display(richness_stats.round(3))
print(f"Richness Spearman ρ = {spearman_richness_r:.3f} (p={spearman_richness_p:.3e})")

## Cell 14: condition jointly on stellar mass and richness

Both observables are split at their sample medians, producing four transparent 2D cells. The cut values, cell sizes, medians, and widths are reported explicitly.

In [ ]:
mstar_median_cut = analysis["logMstar_z8"].median()
richness_median_cut = analysis["Ngal_R5_z8"].median()
analysis["Mstar_half"] = np.where(analysis["logMstar_z8"] < mstar_median_cut, "low Mstar", "high Mstar")
analysis["Richness_half"] = np.where(analysis["Ngal_R5_z8"] <= richness_median_cut,
                                      "low richness", "high richness")
analysis["Mstar_Richness_cell"] = analysis["Mstar_half"] + " / " + analysis["Richness_half"]

joint_order = [
    "low Mstar / low richness", "low Mstar / high richness",
    "high Mstar / low richness", "high Mstar / high richness",
]
analysis["Mstar_Richness_cell"] = pd.Categorical(
    analysis["Mstar_Richness_cell"], categories=joint_order, ordered=True
)
joint_stats = grouped_distribution_stats(analysis, "Mstar_Richness_cell")

print(f"Median stellar-mass cut : logMstar = {mstar_median_cut:.3f}")
print(f"Median richness cut     : Ngal = {richness_median_cut:g}")
display(joint_stats.round(3))

fig, ax = plt.subplots()
for label, group in analysis.groupby("Mstar_Richness_cell", observed=True):
    ax.hist(group["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
            label=f"{label} (N={len(group)})")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
ax.legend(fontsize=8)
plt.show()

## Cell 15: compare conditional-distribution widths

For a partition into observable bins, the representative conditional width is defined as the sample-count-weighted mean of the per-bin $W_{68}$. The representative median is defined analogously from per-bin medians. Bins with fewer than `min_conditional_n` objects are excluded and reported; this avoids presenting unstable widths as precise constraints. This summary measures typical within-bin scatter, not a formal Bayesian posterior width.

In [ ]:
def aggregate_conditional_stats(stats_table, min_n=min_conditional_n):
    used = stats_table[(stats_table["N"] >= min_n) & stats_table["W68"].notna()].copy()
    if used.empty:
        return np.nan, np.nan, 0, len(stats_table)
    weights = used["N"].to_numpy(dtype=float)
    representative_median = np.average(used["Median"], weights=weights)
    weighted_w68 = np.average(used["W68"], weights=weights)
    return representative_median, weighted_w68, int(weights.sum()), len(stats_table) - len(used)


conditions = [
    ("M8 only", "full mass-selected sample", pd.DataFrame([baseline_stats])),
    ("M8 + Mstar", "Mstar rank tertiles", mstar_stats),
    ("M8 + SFR", "SFR rank tertiles", sfr_stats),
    ("M8 + Mstar + SFR", "Mstar tertiles × within-tertile SFR halves", mstar_sfr_stats),
    ("M8 + Ngal", "Ngal rank tertiles", richness_stats),
    ("M8 + Mstar + Ngal", "median Mstar × median Ngal (2×2)", joint_stats),
]

summary_rows = []
for condition, definition, table in conditions:
    median_rep, width_rep, n_used, n_excluded = aggregate_conditional_stats(table)
    summary_rows.append({
        "Condition": condition,
        "Conditional-bin definition": definition,
        "N used": n_used,
        "Representative median(logM0)": median_rep,
        "Weighted W68 [dex]": width_rep,
        "W68 reduction vs baseline": 1 - width_rep / baseline_stats["W68"],
        "Small bins excluded": n_excluded,
    })
condition_summary = pd.DataFrame(summary_rows)
display(condition_summary.round({
    "Representative median(logM0)": 3,
    "Weighted W68 [dex]": 3,
    "W68 reduction vs baseline": 3,
}))

## Cell 16: compare feature-level predictive information

In [ ]:
feature_map = {
    "M8": "logM200c_z8",
    "central Mstar": "logMstar_z8",
    "central SFR": "logSFR_z8",
    "Ngal": "Ngal_R5_z8",
}
corr_rows = []
for label, feature in feature_map.items():
    subset = analysis[[feature, "logM200c_z0"]].dropna()
    rho, p_value = stats.spearmanr(subset[feature], subset["logM200c_z0"])
    corr_rows.append({"Feature": label, "N": len(subset), "Spearman rho": rho, "p-value": p_value})
correlation_table = pd.DataFrame(corr_rows).sort_values("Spearman rho", key=np.abs, ascending=False)
display(correlation_table.round({"Spearman rho": 3, "p-value": 5}))

rho_mstar = float(correlation_table.loc[correlation_table["Feature"] == "central Mstar", "Spearman rho"].iloc[0])
rho_ngal = float(correlation_table.loc[correlation_table["Feature"] == "Ngal", "Spearman rho"].iloc[0])
stronger_mstar_or_ngal = "central stellar mass" if abs(rho_mstar) > abs(rho_ngal) else "galaxy richness"
print(f"Between Mstar and Ngal, the stronger rank association is: {stronger_mstar_or_ngal}")

## Cell 17: optional simple regression

A Random Forest is used only as a predictive cross-check. All models use the same train/test rows. The split is grouped by `GroupID_z0`, so multiple $z\simeq8$ halos sharing one descendant can never leak across train and test sets. RMSE and MAE are evaluated exclusively on the held-out test set. This does not replace the transparent conditional-distribution analysis above.

In [ ]:
regression_available = False
rf_models = {}
regression_results = pd.DataFrame()

try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import mean_squared_error, mean_absolute_error

    model_features = {
        "M8": ["logM200c_z8"],
        "M8 + Mstar": ["logM200c_z8", "logMstar_z8"],
        "M8 + Mstar + SFR": ["logM200c_z8", "logMstar_z8", "logSFR_z8"],
        "M8 + Mstar + SFR + Ngal": ["logM200c_z8", "logMstar_z8", "logSFR_z8", "Ngal_R5_z8"],
    }
    all_features = model_features["M8 + Mstar + SFR + Ngal"]
    reg_data = analysis.dropna(subset=all_features + ["logM200c_z0"])

    if len(reg_data) < 500:
        print(f"Skipping Random Forest: only {len(reg_data)} complete rows (<500).")
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state)
        train_pos, test_pos = next(splitter.split(reg_data, groups=reg_data["GroupID_z0"]))
        train_idx = reg_data.index[train_pos]
        test_idx = reg_data.index[test_pos]
        rows = []
        for label, features in model_features.items():
            model = RandomForestRegressor(
                n_estimators=400,
                min_samples_leaf=10,
                max_features=1.0,
                n_jobs=-1,
                random_state=random_state,
            )
            model.fit(reg_data.loc[train_idx, features], reg_data.loc[train_idx, "logM200c_z0"])
            pred = model.predict(reg_data.loc[test_idx, features])
            rows.append({
                "Features": label,
                "N train": len(train_idx),
                "N test": len(test_idx),
                "RMSE [dex]": np.sqrt(mean_squared_error(reg_data.loc[test_idx, "logM200c_z0"], pred)),
                "MAE [dex]": mean_absolute_error(reg_data.loc[test_idx, "logM200c_z0"], pred),
            })
            rf_models[label] = (model, features)
        regression_results = pd.DataFrame(rows)
        regression_available = True
        display(regression_results.round({"RMSE [dex]": 3, "MAE [dex]": 3}))
except ImportError as exc:
    print(f"scikit-learn is unavailable; skipping optional regression ({exc}).")

## Cell 18: Random-Forest feature importance

Impurity-based importance is shown only for the full model. It describes how this fitted model used correlated predictors; it is **not** evidence of physical causation.

In [ ]:
if regression_available:
    full_label = "M8 + Mstar + SFR + Ngal"
    full_model, full_features = rf_models[full_label]
    feature_importance = pd.DataFrame({
        "Feature": full_features,
        "Importance": full_model.feature_importances_,
    }).sort_values("Importance", ascending=False)
    display(feature_importance.round({"Importance": 3}))

    fig, ax = plt.subplots()
    ax.barh(feature_importance["Feature"][::-1], feature_importance["Importance"][::-1])
    ax.set(xlabel="Random-Forest impurity importance", title="Predictive diagnostic only")
    plt.show()
else:
    feature_importance = pd.DataFrame(columns=["Feature", "Importance"])
    print("Feature importance not computed because Cell 17 did not fit the model.")

## Cell 19: final comparison figure

These are overlapping, normalized simulation-derived distributions. “High Mstar” means at or above the stellar-mass median, while “high richness” means strictly above the (discrete) richness median defined in Cell 14. The curves demonstrate both shifts and changes in width.

In [ ]:
high_mstar = analysis[analysis["Mstar_half"] == "high Mstar"]
high_richness = analysis[analysis["Richness_half"] == "high richness"]
high_both = analysis[
    (analysis["Mstar_half"] == "high Mstar")
    & (analysis["Richness_half"] == "high richness")
]

final_samples = [
    ("M8 only", desc),
    ("high Mstar", high_mstar),
    ("high Ngal", high_richness),
    ("high Mstar + high Ngal", high_both),
]
fig, ax = plt.subplots(figsize=(8.0, 5.2))
for label, frame in final_samples:
    width = summarize_distribution(frame["logM200c_z0"])["W68"]
    ax.hist(frame["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
            label=f"{label}: N={len(frame)}, W68={width:.2f}")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
ax.legend(fontsize=9)
plt.show()

## Cell 20: automatically generated summary

In [ ]:
def summary_value(condition, column):
    return condition_summary.loc[condition_summary["Condition"] == condition, column].iloc[0]


baseline_w68 = float(baseline_stats["W68"])
mstar_w68 = summary_value("M8 + Mstar", "Weighted W68 [dex]")
sfr_w68 = summary_value("M8 + SFR", "Weighted W68 [dex]")
mstar_sfr_w68 = summary_value("M8 + Mstar + SFR", "Weighted W68 [dex]")
ngal_w68 = summary_value("M8 + Ngal", "Weighted W68 [dex]")
joint_w68 = summary_value("M8 + Mstar + Ngal", "Weighted W68 [dex]")
joint_reduction = 1 - joint_w68 / baseline_w68

if regression_available:
    best_reg = regression_results.loc[regression_results["RMSE [dex]"].idxmin()]
    regression_sentence = (
        f"The best optional held-out Random-Forest model was **{best_reg['Features']}** "
        f"with RMSE={best_reg['RMSE [dex]']:.3f} dex and MAE={best_reg['MAE [dex]']:.3f} dex."
    )
else:
    regression_sentence = "The optional Random-Forest comparison was not available and is not used in the conclusions."

display(Markdown(fr"""
### Summary

1. Knowing only $10^{{{logM_min:.1f}}}<M_8/M_\odot<10^{{{logM_max:.1f}}}$ gives
   $W_{{68}}={baseline_w68:.3f}$ dex for the $z=0$ FoF host mass.
2. Conditioning on central stellar-mass tertile gives a typical within-bin width of
   **{mstar_w68:.3f} dex** ({1-mstar_w68/baseline_w68:.1%} change relative to baseline).
3. Conditioning on SFR tertile gives **{sfr_w68:.3f} dex**; conditioning on SFR within stellar-mass tertiles gives
   **{mstar_sfr_w68:.3f} dex**.
4. Conditioning on richness tertile gives **{ngal_w68:.3f} dex**. Joint median-split $M_\star+N_{{\rm gal}}$
   cells give **{joint_w68:.3f} dex**, a **{joint_reduction:.1%}** reduction relative to baseline.
5. Between central stellar mass and richness, **{stronger_mstar_or_ngal}** has the stronger absolute Spearman association
   with descendant mass in this sample ($\rho_{{Mstar}}={rho_mstar:.3f}$; $\rho_{{Ngal}}={rho_ngal:.3f}$).
6. {regression_sentence}
7. For a future JWST-facing analysis, stellar-mass proxies, SFR/UV indicators, and projected richness are the direct next
   observables to test. A realistic extension must forward-model projection, selection completeness, and measurement errors.

All quoted widths are simulation-derived conditional-distribution widths. They are not formal Bayesian posterior credible intervals,
and small conditional cells should not be over-interpreted.
"""))